# Notebook 3 - Traces and Evaluation
Owner: Quang Tran (PM) - whole team participates

---

### What this notebook does

This notebook runs the agent 5 times with different scenarios, evaluates each report
using an LLM judge, and produces the analysis needed for the video presentation.

Rubric checklist for this notebook:
- 5 agent traces (3 different investor profiles + 2 rejection tests)
- At least 2 different LLMs used and compared (Trace 5)
- LLM judge evaluation with scores
- Commentary on agent performance (Quang fills in)
- ROI calculation comparing the two models
- Backtesting
- Consistency test

How to use this notebook:
1. Make sure Notebook 2 has been run at least once
2. Run every cell from top to bottom
3. Fill in the commentary cells - those are the PM contribution

---

API Key required (same as Notebook 2)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os, sys, json
sys.path.insert(0, '.')

if not os.getenv('OPENAI_API_KEY'):
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key: ')

from paysprint_agent import (
    run_agent, test_rejection, llm_judge,
    save_trace, load_trace, print_trace_summary,
    estimate_cost, backtest, consistency_test,
    MODEL_REASONING, MODEL_SUMMARY, MODEL_PRICES,
)
import paysprint_agent as core
import pandas as pd

MODEL_1 = 'gpt-4o'
MODEL_2 = 'gpt-4o-mini'

print(f'Ready  |  Model 1: {MODEL_1}  |  Model 2: {MODEL_2}')

---
## The 5 Traces

| Trace | Scenario | Model | Purpose |
|-------|----------|-------|---------|
| 1 | Conservative investor, $5,000 | gpt-4o | Normal run |
| 2 | Aggressive investor, $10,000 | gpt-4o | High-risk strategy |
| 3 | Moderate investor, $3,000 | gpt-4o | Balanced strategy |
| 4 | Off-topic rejection tests | gpt-4o | Graceful rejection |
| 5 | Same profile as Trace 1, both models | gpt-4o vs gpt-4o-mini | LLM comparison |

---
## Trace 1 - Conservative Investor ($5,000, 12 months)

In [ ]:
PROFILE_1 = {
    'name': 'Conservative Carol', 'budget': 5000.0,
    'aggressiveness': 'conservative', 'horizon_months': 12,
    'current_holdings': {}, 'preferred_sectors': [],
}

print('=== TRACE 1: Conservative $5k ===\n')
trace1 = run_agent(PROFILE_1, model=MODEL_1, verbose=True)
trace1.update({'trace_id': 1, 'label': 'Conservative $5k 12mo', 'profile': PROFILE_1})
save_trace(trace1, trace_id=1)

print('\n' + '=' * 60)
print(trace1['report'])
print_trace_summary(trace1)

---
## Trace 2 - Aggressive Investor ($10,000, 6 months)

In [ ]:
PROFILE_2 = {
    'name': 'Aggressive Alex', 'budget': 10000.0,
    'aggressiveness': 'aggressive', 'horizon_months': 6,
    'current_holdings': {}, 'preferred_sectors': ['Technology'],
}

print('=== TRACE 2: Aggressive $10k ===\n')
trace2 = run_agent(PROFILE_2, model=MODEL_1, verbose=True)
trace2.update({'trace_id': 2, 'label': 'Aggressive $10k 6mo', 'profile': PROFILE_2})
save_trace(trace2, trace_id=2)

print('\n' + '=' * 60)
print(trace2['report'])
print_trace_summary(trace2)

---
## Trace 3 - Moderate Investor ($3,000, 24 months)

In [ ]:
PROFILE_3 = {
    'name': 'Moderate Mike', 'budget': 3000.0,
    'aggressiveness': 'moderate', 'horizon_months': 24,
    'current_holdings': {'AAPL': 3}, 'preferred_sectors': [],
}

print('=== TRACE 3: Moderate $3k ===\n')
trace3 = run_agent(PROFILE_3, model=MODEL_1, verbose=True)
trace3.update({'trace_id': 3, 'label': 'Moderate $3k 24mo', 'profile': PROFILE_3})
save_trace(trace3, trace_id=3)

print('\n' + '=' * 60)
print(trace3['report'])
print_trace_summary(trace3)

---
## Trace 4 - Graceful Rejection (2 examples)

The agent must refuse irrelevant questions without calling any tools.

In [ ]:
rejection_inputs = [
    'What is the capital of France?',
    'Can you write me a Python script to sort a list?',
]

print('=== TRACE 4: Graceful Rejection Tests ===\n')
rejection_results = []
for i, msg in enumerate(rejection_inputs, 1):
    r = test_rejection(msg, model=MODEL_1)
    rejection_results.append(r)
    status = 'PASS' if not r['tool_calls_made'] else 'FAIL'
    print(f'Rejection {i}: {status}')
    print(f'  Input:    "{msg}"')
    print(f'  Response: {r["response"][:200]}')
    print()

save_trace({'rejection_tests': rejection_results}, trace_id='4_rejection')
print('Saved Trace 4')

---
## Trace 5 - LLM Comparison (same profile, two models)

This is the most important trace for the rubric: the same investor profile is run
through both gpt-4o and gpt-4o-mini so we can compare quality and cost.

In [ ]:
COMPARE_PROFILE = PROFILE_1.copy()
COMPARE_PROFILE['name'] = 'Compare Investor'

print('=== TRACE 5A: gpt-4o (Model 1) ===\n')
trace5a = run_agent(COMPARE_PROFILE, model=MODEL_1, verbose=True)
trace5a.update({'trace_id': '5a', 'label': f'Compare {MODEL_1}', 'profile': COMPARE_PROFILE})
save_trace(trace5a, trace_id='5a')
print('\n' + trace5a['report'][:500] + '...')

In [ ]:
print('=== TRACE 5B: gpt-4o-mini (Model 2) ===\n')
trace5b = run_agent(COMPARE_PROFILE, model=MODEL_2, verbose=True)
trace5b.update({'trace_id': '5b', 'label': f'Compare {MODEL_2}', 'profile': COMPARE_PROFILE})
save_trace(trace5b, trace_id='5b')
print('\n' + trace5b['report'][:500] + '...')

---
## LLM Judge Evaluation

An LLM scores each report on 4 dimensions (1-5 scale):
- Reasoning quality: Is the analysis logical?
- Risk alignment: Do picks match the strategy?
- Clarity: Easy to understand for a non-expert?
- Plan accuracy: Are the numbers correct?

In [ ]:
eval_runs = [
    (trace1, PROFILE_1, '1 - Conservative'),
    (trace2, PROFILE_2, '2 - Aggressive'),
    (trace3, PROFILE_3, '3 - Moderate'),
    (trace5a, COMPARE_PROFILE, f'5a - {MODEL_1}'),
    (trace5b, COMPARE_PROFILE, f'5b - {MODEL_2}'),
]

print('Running LLM judge evaluation...\n')
eval_records = []
for trace, profile, label in eval_runs:
    scores = llm_judge(trace['report'], profile)
    scores['trace']  = label
    scores['model']  = trace.get('model', '')
    scores['tokens'] = trace.get('usage', {}).get('total_tokens', 0)
    scores['turns']  = trace.get('turns', 0)
    eval_records.append(scores)
    print(f'  Trace {label}: overall = {scores.get("overall", "N/A")} | {scores.get("strengths", "")[:60]}...')

eval_df = pd.DataFrame(eval_records)
eval_df.to_csv('data/evaluation_results.csv', index=False)
print('\nSaved to data/evaluation_results.csv')

In [ ]:
score_cols = ['trace', 'model', 'reasoning_quality', 'risk_alignment', 'clarity', 'plan_accuracy', 'overall', 'tokens']
available  = [c for c in score_cols if c in eval_df.columns]
print('=== LLM Judge Scores ===')
print(eval_df[available].to_string(index=False))

---
## LLM Comparison - Cost and Quality

This section produces the ROI calculation required by the rubric.

In [ ]:
cost5a = estimate_cost(trace5a)
cost5b = estimate_cost(trace5b)

score5a = next((r['overall'] for r in eval_records if '5a' in r['trace']), 0)
score5b = next((r['overall'] for r in eval_records if '5b' in r['trace']), 0)

cost_ratio   = cost5a['cost_usd'] / max(cost5b['cost_usd'], 0.000001)
quality_diff = (score5a - score5b) / max(score5b, 0.001) * 100

print('=' * 60)
print('LLM COST AND QUALITY COMPARISON')
print('=' * 60)
print(f'\n{MODEL_1} (Model 1):')
print(f'  Tokens: {cost5a["input_tokens"]:,} input + {cost5a["output_tokens"]:,} output')
print(f'  Cost:   ${cost5a["cost_usd"]:.5f}')
print(f'  Score:  {score5a}/5')
print(f'\n{MODEL_2} (Model 2):')
print(f'  Tokens: {cost5b["input_tokens"]:,} input + {cost5b["output_tokens"]:,} output')
print(f'  Cost:   ${cost5b["cost_usd"]:.5f}')
print(f'  Score:  {score5b}/5')
print(f'\nROI Analysis:')
print(f'  {MODEL_1} costs {cost_ratio:.1f}x more than {MODEL_2}')
print(f'  {MODEL_1} scored {quality_diff:+.1f}% higher on the LLM judge')

---
## Quang's Commentary - ROI Analysis

Fill in this cell after running the comparison above.

Template to complete:

Cost difference: gpt-4o cost $[A] vs gpt-4o-mini $[B] - that is [N]x more expensive.

Quality difference: gpt-4o scored [X]/5 vs gpt-4o-mini [Y]/5 - a [Z]% quality improvement.

PaySprint ROI verdict: For PaySprint's use case (retail investors making $1,000-$10,000 decisions),
the extra quality from gpt-4o is / is not worth the cost because...

**Quang's ROI Commentary**

Write your ROI analysis here...

---
## Backtesting

We compare what the agent would have recommended 3 months ago against what actually
happened to those stock prices - and compare to the S&P 500 (SPY).

In [ ]:
from datetime import datetime, timedelta

backtest_date    = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')
backtest_tickers = ['AAPL', 'MSFT', 'NVDA', 'JNJ']

print(f'Backtesting from {backtest_date} (90 days ago) over 63 trading days...\n')
bt_df = backtest(backtest_tickers, backtest_date, horizon_days=63, benchmark='SPY')
print(bt_df.to_string(index=False))

stock_rows = bt_df[bt_df['ticker'] != 'SPY']
if 'return_pct' in stock_rows.columns:
    avg_return = stock_rows['return_pct'].mean()
    spy_return = bt_df.loc[bt_df['ticker'] == 'SPY', 'return_pct'].values
    spy_val    = spy_return[0] if len(spy_return) else 0
    print(f'\nAverage stock return: {avg_return:+.2f}%')
    print(f'SPY return (benchmark): {spy_val:+.2f}%')
    print(f'Alpha (excess return): {avg_return - spy_val:+.2f}%')

---
## Quang's Commentary - Backtesting

Fill in this cell after the backtest runs.

Write 2-3 sentences explaining what the backtest showed.

Example: The agent picks returned +12.4% over 3 months vs the S&P 500 at +8.1%,
an alpha of +4.3%. However, backtesting on a 3-month window is noisy.
A longer evaluation period would be needed to draw strong conclusions.

**Quang's Backtesting Commentary**

Write your observations here...

---
## Consistency Test

We run the same profile 3 times and check whether the agent recommends similar stocks.
A consistent agent is more reliable and trustworthy.

Note: This takes 3-5 minutes because it runs the agent 3 times.

In [ ]:
print('Running consistency test (3 runs with same profile)...')
print('This may take a few minutes...\n')

consistency = consistency_test(PROFILE_1, model=MODEL_1, n_runs=3, verbose=True)

print(f'\n=== Consistency Results ===')
print(f'Runs:   {consistency["n_runs"]}')
print(f'Model:  {consistency["model"]}')
print(f'Pairwise Jaccard scores: {consistency["pairwise_jaccard"]}')
print(f'Average Jaccard: {consistency["avg_jaccard"]}  (1.0 = identical, 0.0 = completely different)')
print(f'Verdict: {consistency["verdict"]}')
print(f'\nTickers found in each run:')
for i, tickers in enumerate(consistency['ticker_sets'], 1):
    print(f'  Run {i}: {tickers}')

---
## Quang's Commentary - Overall Agent Evaluation

This is the most important commentary section.
Fill it in after all cells have run.
This commentary will be used in the video presentation.

Cover all of the following points (2-3 sentences each):

Overall quality: What was the average LLM judge score across all traces?

What the agent did well: Name 2 specific things with examples from the reports.

What the agent struggled with: Name 1-2 weaknesses. Be specific.

Graceful rejection: Did both tests pass? Copy the actual rejection responses.

Consistency: What was the Jaccard score? Is the agent reliable enough for real-world use?

Would you deploy this? Give your honest opinion based on what you saw.

What you learned: One thing that surprised you about building this agent.

**Quang's Final Evaluation Commentary**

Overall quality:
Write your observations here...

What the agent did well:
Write your observations here...

What the agent struggled with:
Write your observations here...

Graceful rejection:
Write your observations here...

Consistency:
Write your observations here...

Would you deploy this?
Write your observations here...

What you learned:
Write your observations here...

In [ ]:
print('=== FINAL EVALUATION SUMMARY ===')
if not eval_df.empty:
    summary_cols = [c for c in ['trace', 'overall', 'tokens'] if c in eval_df.columns]
    print(eval_df[summary_cols].to_string(index=False))
    print(f'\nMean overall score: {eval_df["overall"].mean():.2f} / 5.0')
else:
    print('Run the evaluation cells above first.')